In [13]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *
from lib.kpi_processor.KPIReportEurope import *
from datetime import date
from datetime import timedelta

In [15]:
customers = Query(query = "SELECT Name FROM KPI_Customer").execute([KPIHub_Conn])
customers

,Name
0,Wales and West Utilities
1,APRETIGAS
2,ADRIGAS
3,EDMA
4,M Reti
...,...
74,Multiservizi Azzanese
75,AMAG Reti
76,AMGAS Foggia
77,RETI Distribuzione


In [16]:
# Run for all countries in KPI_Country table

countries = Query("SELECT Name FROM KPI_Country").execute([KPIHub_Conn])['Name'].tolist()

for country in countries:
    print(f"Processing: {country}")
    print(f"    KPIReport")
    kpi_r = KPIReport(process_dict={'Country': country}, period_dict={})
    kpi_r.query_table()
    kpi_r.process_data()
    kpi_r.push_data()
    print(f"    KPIEmissionSource")
    kpi_es = KPIEmissionSource(process_dict={'Country': country}, period_dict={})
    kpi_es.query_table()
    kpi_es.process_data()
    kpi_es.push_data()

customers = Query("SELECT Name FROM KPI_Customer").execute([KPIHub_Conn])['Name'].tolist()

for customer in customers:
    print(f"Processing: {customer}")
    print(f"    KPIReport")
    kpi_r = KPIReport(process_dict={'Customer': customer}, period_dict={})
    kpi_r.query_table()
    kpi_r.process_data()
    kpi_r.push_data()
    print(f"    KPIEmissionSource")
    kpi_es = KPIEmissionSource(process_dict={'Customer': customer}, period_dict={})
    kpi_es.query_table()
    kpi_es.process_data()
    kpi_es.push_data()


Processing: United Kingdom
    KPIReport
    KPIEmissionSource


Processing: Italy
    KPIReport
    KPIEmissionSource
Processing: Romania
    KPIReport
    KPIEmissionSource
Processing: Switzerland
    KPIReport
    KPIEmissionSource
Processing: Greece
    KPIReport
    KPIEmissionSource
Processing: Germany
    KPIReport
    KPIEmissionSource
Processing: Poland
    KPIReport
    KPIEmissionSource
Processing: Austria
    KPIReport
    KPIEmissionSource
Processing: Ireland
    KPIReport
    KPIEmissionSource
Processing: Czechia
    KPIReport
No data to push for Czechia
    KPIEmissionSource
No data to push for Czechia
Processing: AZ
    KPIReport
    KPIEmissionSource
Processing: Netherlands
    KPIReport
    KPIEmissionSource
Processing: Wales and West Utilities
    KPIReport
    KPIEmissionSource
Processing: APRETIGAS
    KPIReport
    KPIEmissionSource
Processing: ADRIGAS
    KPIReport
    KPIEmissionSource
Processing: EDMA
    KPIReport
No data to push for EDMA
    KPIEmissionSource
No data to push for EDMA
Processing: M Reti
    KPIReport
    KP

In [17]:
query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CountryId FROM KPI_Country WHERE Name = 'United Kingdom' AND KPIId = 'LisaDensity')"
Query(query = query).execute([KPIHub_Conn])

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,LisaDensity_United Kingdom_Y2025,LisaDensity,00000000-0000-0000-0000-000000000001,None,2025,Year,None,2.12,None,2026-08-14 11:17:18.418781
1,LisaDensity_United Kingdom_Y2026,LisaDensity,00000000-0000-0000-0000-000000000001,None,2026,Year,None,1.28,None,2026-08-14 11:17:18.418781


In [18]:
query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Wales and West Utilities' AND KPIId = 'LisaDensity')"
Query(query = query).execute([KPIHub_Conn])

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,LisaDensity_Wales and West Utilities_Y2025,LisaDensity,027114F8-DDB7-C0D0-1398-3A173A08C9BE,None,2025,Year,None,3.96,None,2026-08-14 11:17:20.924463
1,LisaDensity_Wales and West Utilities_Y2026,LisaDensity,027114F8-DDB7-C0D0-1398-3A173A08C9BE,None,2026,Year,None,1.98,None,2026-08-14 11:17:20.924463
